In [1]:
!pip -q install vaderSentiment
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np
BASE = '/content/drive/MyDrive/CreatorPulse'
RAW, SILVER, DASH = f'{BASE}/01_Data/bronze_raw', f'{BASE}/01_Data/silver_clean', f'{BASE}/04_Dashboard'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.5 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
com = pd.read_csv(f'{RAW}/comments.csv')
print('comments.csv rows:', len(com))
print('columns:', list(com.columns))
print('\nMissing values:\n', com.isna().sum())
lab = [c for c in com.columns if 'sentiment' in c.lower() or 'label' in c.lower()]
for c in lab:
    print(f'\nClass balance for {c}:')
    print(com[c].value_counts(normalize=True).round(3))
display(com.head(3))

comments.csv rows: 18409
columns: ['Unnamed: 0', 'Video ID', 'Comment', 'Likes', 'Sentiment']

Missing values:
 Unnamed: 0    0
Video ID      0
Comment       1
Likes         0
Sentiment     0
dtype: int64

Class balance for Sentiment:
Sentiment
2.0    0.621
1.0    0.252
0.0    0.127
Name: proportion, dtype: float64


,Unnamed: 0,Video ID,Comment,Likes,Sentiment
0,0,wAZZ-UWGVHI,Let's not forget that Apple Pay in 2014 requir...,95.0,1.0
1,1,wAZZ-UWGVHI,Here in NZ 50% of retailers don’t even have co...,19.0,0.0
2,2,wAZZ-UWGVHI,I will forever acknowledge this channel with t...,161.0,2.0


In [3]:
vs = pd.read_csv(f'{RAW}/videos-stats.csv')
print('rows:', len(vs), '| columns:', list(vs.columns))
lk = next((c for c in vs.columns if c.lower() == 'likes'), None)
cm = next((c for c in vs.columns if c.lower() == 'comments'), None)
if lk and cm:
    hidden = (vs[lk] == -1) | (vs[cm] == -1)
    print('Rows with hidden likes or comments (-1):', int(hidden.sum()))
    print('Rows after removing them:', int((~hidden).sum()))
else:
    print('Could not find the Likes / Comments columns. Columns are listed above.')

rows: 1881 | columns: ['Unnamed: 0', 'Title', 'Video ID', 'Published At', 'Keyword', 'Likes', 'Comments', 'Views']
Rows with hidden likes or comments (-1): 9
Rows after removing them: 1872


In [4]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
sr = pd.read_csv(f'{SILVER}/sentiment_results.csv')
print('Rows:', len(sr), '| columns:', list(sr.columns))
acc = accuracy_score(sr.true_label, sr.pred_label)
base = sr.true_label.value_counts(normalize=True).max()
mf1 = f1_score(sr.true_label, sr.pred_label, average='macro')
print(f'\nBaseline (always predict the majority class): {base:.1%}')
print(f'VADER accuracy: {acc:.1%}')
print(f'VADER macro F1: {mf1:.3f}\n')
print(classification_report(sr.true_label, sr.pred_label, digits=3))
labs = ['negative', 'neutral', 'positive']
cmx = pd.DataFrame(confusion_matrix(sr.true_label, sr.pred_label, labels=labs),
                   index=[f'actual {l}' for l in labs], columns=[f'pred {l}' for l in labs])
display(cmx)

Rows: 18644 | columns: ['video_id', 'title', 'keyword', 'comment', 'comment_likes', 'true_label', 'pred_label', 'vader_score']

Baseline (always predict the majority class): 62.2%
VADER accuracy: 71.7%
VADER macro F1: 0.597

              precision    recall  f1-score   support

    negative      0.481     0.518     0.499      2352
     neutral      0.648     0.353     0.457      4696
    positive      0.775     0.905     0.835     11596

    accuracy                          0.717     18644
   macro avg      0.635     0.592     0.597     18644
weighted avg      0.706     0.717     0.697     18644



,pred negative,pred neutral,pred positive
actual negative,1219,326,807
actual neutral,795,1658,2243
actual positive,521,575,10500


In [5]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
scores = sr['comment'].astype(str).apply(lambda t: sia.polarity_scores(t)['compound'])
pred = np.where(scores >= 0.05, 'positive', np.where(scores <= -0.05, 'negative', 'neutral'))
print('Same label as the saved file:', f'{(pred == sr.pred_label).mean():.1%}')
print('Largest score difference vs saved vader_score:', round(float((scores - sr.vader_score).abs().max()), 4))
print('Accuracy of this re-run:', f'{accuracy_score(sr.true_label, pred):.1%}')

Same label as the saved file: 100.0%
Largest score difference vs saved vader_score: 0.0
Accuracy of this re-run: 71.7%


In [6]:
kw = (sr.groupby('keyword')['true_label'].value_counts(normalize=True).unstack(fill_value=0) * 100).round(1)
saved = pd.read_csv(f'{SILVER}/sentiment_by_keyword.csv').set_index('keyword')
cols = ['negative', 'neutral', 'positive']
diff = (kw[cols] - saved[cols]).abs().max().max()
print('Topics here:', len(kw), '| in saved table:', len(saved))
print('Largest difference vs saved table (percentage points):', round(float(diff), 2))
display(kw.sort_values('positive', ascending=False).head(5))

Topics here: 41 | in saved table: 41
Largest difference vs saved table (percentage points): 0.0


true_label,negative,neutral,positive
keyword,,,
lofi,2.3,12.2,85.5
machine learning,3.3,18.9,77.8
mathchemistry,6.8,15.8,77.4
asmr,2.9,20.2,76.8
music,2.5,21.1,76.4


In [7]:
err = sr[(sr.pred_label == 'positive') & (sr.true_label == 'negative')]
print('VADER said positive but the label is negative:', len(err), 'comments. Five examples:\n')
for t in err.sample(5, random_state=1)['comment']:
    print('-', str(t)[:200])
neu = sr[sr.true_label == 'neutral']
print(f'\nRecall on neutral comments: {(neu.pred_label == "neutral").mean():.1%}')
for kwd in ['cnn', 'lofi']:
    d = sr[sr.keyword == kwd]
    print(f"{kwd}: dataset labels {(d.true_label == 'positive').mean():.1%} positive, VADER {(d.pred_label == 'positive').mean():.1%} positive")

VADER said positive but the label is negative: 807 comments. Five examples:

- I think the pandemic has taught people the importance of multiple streams of income unfortunately having a job doesn’t mean security
- Humans: I feel afraid.
AI: I feel like I'm falling forward into an unknown future that holds great danger.
- I’ve been trying to grow my portfolio of $460k for sometime now, my major challenge is not knowing when to sell or hold.
- Aliens are intelligent enough to see that our governments are doing a good enough job at destroying the world
- Jane was so but hurt over "disrespecting" a brother 😂😂😂💀💀

Recall on neutral comments: 35.3%
cnn: dataset labels 14.9% positive, VADER 47.9% positive
lofi: dataset labels 85.5% positive, VADER 85.0% positive


In [8]:
pd.DataFrame({'metric': ['rows', 'baseline_accuracy', 'vader_accuracy', 'vader_macro_f1'],
              'value': [len(sr), round(base, 4), round(acc, 4), round(mf1, 4)]}).to_csv(f'{DASH}/sentiment_evaluation.csv', index=False)
print('Saved 04_Dashboard/sentiment_evaluation.csv')

Saved 04_Dashboard/sentiment_evaluation.csv
